In [2]:
import networkx as nx
import numpy as np
import pandas as pd
import time
import random
from collections import defaultdict, Counter

def load_graph(path):
    if path.endswith('.csv'):
        try:
            df = pd.read_csv(path, low_memory=False)
            if len(df.columns) < 2:
                df = pd.read_csv(path, sep=None, engine='python')

            if len(df.columns) < 2:
                raise ValueError("Dataset does not have at least 2 columns.")

            source_col = df.columns[0]
            target_col = df.columns[1]
            df = df.dropna(subset=[source_col, target_col])

            G = nx.from_pandas_edgelist(df, source=source_col, target=target_col)
        except Exception:
            G = nx.read_edgelist(path, delimiter=',', data=False, comments='#')
    else:
        G = nx.read_edgelist(path, comments='#')

    G.remove_edges_from(nx.selfloop_edges(G))
    G = nx.convert_node_labels_to_integers(G)
    return G

def ANCE_Overlapping(G, x_dict, Atr, precomp_neighbors):
    """
    Optimized Algorithm 1 + Algorithm 3
    """
    L1_edges = []
    probabilities = {}

    for node in G.nodes():
        neighbors = precomp_neighbors[node]
        d_i = len(neighbors)
        if d_i == 0:
            probabilities[node] = []
            continue

        x_i = x_dict[node]

        # Inlined math functions for speed
        h_i = 1 / (1 + np.exp(-np.clip(x_i, -500, 500)))
        e_x = np.exp(h_i - np.max(h_i))
        p_i = e_x / e_x.sum()

        p_i_edges = p_i[:-1]
        s1_idx = np.argmax(p_i_edges)
        L1_edges.append((node, neighbors[s1_idx]))

        probabilities[node] = list(zip(neighbors, p_i_edges))

    G1 = nx.Graph()
    G1.add_nodes_from(G.nodes())
    G1.add_edges_from(L1_edges)

    base_labels = {node: min(comp) for comp in nx.connected_components(G1) for node in comp}

    overlapping_communities = defaultdict(set)
    for node, c_id in base_labels.items():
        overlapping_communities[c_id].add(node)

    # MASSIVE SPEEDUP 1: Precalculate the dominant attribute for each base community ONCE.
    comm_dominant_attr = {}
    for c_id, nodes in overlapping_communities.items():
        attrs = [Atr[n] for n in nodes]
        comm_dominant_attr[c_id] = Counter(attrs).most_common(1)[0][0]

    for node in G.nodes():
        if node not in probabilities: continue

        neighbors = precomp_neighbors[node]
        d_i = len(neighbors)
        if d_i == 0: continue

        prob_threshold = 1.0 / d_i

        for neighbor, prob in probabilities[node]:
            if prob > prob_threshold:
                neighbor_comm = base_labels[neighbor]
                if neighbor_comm != base_labels[node]:
                    # O(1) lookup instead of O(N) list building and Counter call
                    if Atr[node] == comm_dominant_attr.get(neighbor_comm):
                        overlapping_communities[neighbor_comm].add(node)

    final_communities = [list(comp) for comp in overlapping_communities.values() if len(comp) > 0]
    return final_communities

def compute_overlapping_objectives(communities, Atr, precomp_neighbors, degrees, m):
    """Evaluates Extended EQ and Attribute Penalty (Optimized)"""
    if m == 0 or len(communities) <= 1:
        return np.array([1.0, 1.0])

    node_comm_counts = defaultdict(int)
    for comm in communities:
        for node in comm:
            node_comm_counts[node] += 1

    eq = 0.0

    # MASSIVE SPEEDUP 2: Optimized edge counting without copying nx.subgraphs
    for comm in communities:
        comm_nodes = set(comm)
        actual_edges_sum = 0.0

        # Check edges using fast set intersection
        for u in comm_nodes:
            cu = node_comm_counts[u]
            for v in precomp_neighbors[u]:
                if v in comm_nodes:
                    cv = node_comm_counts[v]
                    # We add 1.0 instead of 2.0 because the loop hits each edge twice (u->v and v->u)
                    actual_edges_sum += 1.0 / (cu * cv)

        expected_sum_part = sum(degrees[u] / node_comm_counts[u] for u in comm_nodes)
        expected_edges_sum = (expected_sum_part ** 2) / (2.0 * m)

        eq += actual_edges_sum - expected_edges_sum

    f1 = - (eq / (2.0 * m))

    f2_penalty = 0.0
    count = 0
    for comm in communities:
        length = len(comm)
        if length > 1:
            attrs = [Atr[n] for n in comm]
            most_common_count = Counter(attrs).most_common(1)[0][1]
            f2_penalty += (length - most_common_count)
            count += length

    f2 = f2_penalty / (count + 1e-6)

    return np.array([f1, f2])

def DE_crossover_mutation(x1, x2, x3, F_weight=0.7, CR=0.4):
    y = {}
    for node in x1.keys():
        mutant = x1[node] + F_weight * (x2[node] - x3[node])
        mask = np.random.rand(len(mutant)) < CR
        if not np.any(mask):
            mask[np.random.randint(len(mask))] = True
        offspring = np.where(mask, mutant, x1[node])

        pm_mask = np.random.rand(len(offspring)) < 0.01
        noise = np.random.normal(0, 1.0, len(offspring))
        offspring = np.where(pm_mask, offspring + noise, offspring)

        y[node] = np.clip(offspring, -10.0, 10.0)
    return y

def CEMOV_Overlapping(G, Atr, pop_size=10, generations=5):
    # MASSIVE SPEEDUP 3: Cache NetworkX generators to Python structures ONCE
    precomp_neighbors = {n: list(G.neighbors(n)) for n in G.nodes()}
    degrees = dict(G.degree())
    m = G.number_of_edges()

    # Generate continuous encodings
    population = []
    for node in G.nodes():
        # Using precomputed neighbor lengths
        pass
    population = [{node: np.random.uniform(-10, 10, len(precomp_neighbors[node]) + 1) for node in G.nodes()} for _ in range(pop_size)]

    weights = np.array([[i/(pop_size-1), 1.0 - i/(pop_size-1)] for i in range(pop_size)])

    t_size = max(2, pop_size // 3)
    B = [[(i+j)%pop_size for j in range(-t_size//2, t_size//2 + 1)] for i in range(pop_size)]

    obj_values = np.zeros((pop_size, 2))
    for i in range(pop_size):
        communities = ANCE_Overlapping(G, population[i], Atr, precomp_neighbors)
        obj_values[i] = compute_overlapping_objectives(communities, Atr, precomp_neighbors, degrees, m)

    z_star = np.min(obj_values, axis=0)

    for gen in range(generations):
        for i in range(pop_size):
            pool = B[i] if np.random.rand() < 0.9 else list(range(pop_size))
            if len(pool) >= 3:
                idx1, idx2, idx3 = np.random.choice(pool, 3, replace=False)
            else:
                idx1, idx2, idx3 = np.random.choice(list(range(pop_size)), 3, replace=False)

            y = DE_crossover_mutation(population[idx1], population[idx2], population[idx3])

            communities_y = ANCE_Overlapping(G, y, Atr, precomp_neighbors)
            obj_y = compute_overlapping_objectives(communities_y, Atr, precomp_neighbors, degrees, m)

            z_star = np.minimum(z_star, obj_y)

            c_r = 0
            n_r = 2

            for j in pool:
                g_old = np.max(weights[j] * np.abs(obj_values[j] - z_star))
                g_new = np.max(weights[j] * np.abs(obj_y - z_star))

                if g_new < g_old:
                    population[j] = y
                    obj_values[j] = obj_y
                    c_r += 1
                if c_r >= n_r:
                    break

    best_idx = np.argmin(obj_values[:, 0])
    best_eq = -obj_values[best_idx, 0] # Return positive Overlapping Modularity (EQ)

    best_individual = population[best_idx]
    best_communities = ANCE_Overlapping(G, best_individual, Atr, precomp_neighbors)
    num_communities = len(best_communities)

    return best_eq, num_communities

# --- Execution Block ---
datasets = [
    "/content/Dataset_CyberCrime_Sean.csv",
    "/content/london_crime_by_lsoa.csv",
    "/content/twitter_combined.txt.gz",
    "/content/Meetings.csv",
    "/content/Phone_Calls.csv",
    "/content/Roles.csv",
    "/content/Sicilian Mafia.csv",
    "/content/facebook_combined.txt.gz"
]

print(f"{'Dataset':<32} | {'Nodes':<6} | {'Edges':<7} | {'Comms':<6} | {'Modularity':<10} | {'Runtime (s)':<12}")
print("-" * 88)

for ds in datasets:
    try:
        start_time = time.time()

        G = load_graph(ds)
        nodes_count = G.number_of_nodes()
        edges_count = G.number_of_edges()

        if nodes_count == 0 or edges_count == 0:
            print(f"{ds.split('/')[-1][:30]:<32} | {nodes_count:<6} | {edges_count:<7} | {'-':<6} | {'Graph Empty':<10} | {'N/A':<12}")
            continue

        # Generates random discrete attributes for testing
        Atr = {node: random.randint(0, 2) for node in G.nodes()}

        best_modularity, num_communities = CEMOV_Overlapping(G, Atr, pop_size=10, generations=5)

        runtime = time.time() - start_time
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {nodes_count:<6} | {edges_count:<7} | {num_communities:<6} | {best_modularity:<10.4f} | {runtime:<12.4f}")

    except FileNotFoundError:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {'N/A':<10} | {'N/A':<12}")
    except Exception as e:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {str(e)[:15]:<10} | {'N/A':<12}")

Dataset                          | Nodes  | Edges   | Comms  | Modularity | Runtime (s) 
----------------------------------------------------------------------------------------
Dataset_CyberCrime_Sean.csv      | 134    | 160     | 21     | 0.6039     | 0.3520      
london_crime_by_lsoa.csv         | 4868   | 4835    | 33     | 0.9676     | 39.2165     
twitter_combined.txt.gz          | 81306  | 1342296 | 1274   | 0.1609     | 1281.1538   
Meetings.csv                     | 95     | 247     | 9      | 0.4686     | 0.3901      
Phone_Calls.csv                  | 92     | 119     | 7      | 0.5838     | 0.4072      
Roles.csv                        | 106    | 80      | 26     | 0.8222     | 0.5073      
Sicilian Mafia.csv               | 143    | 325     | 13     | 0.3845     | 0.5613      
facebook_combined.txt.gz         | 4039   | 88234   | 113    | 0.3593     | 18.1019     
